In [ ]:
#!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
#!pip install -q PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00


User Query
→ Convert to embedding

→ Retrieve relevant chunks (vector DB)

→ Add to prompt

→ LLM generates answer

# Import Libraries

In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import  pipeline

c:\Users\Pratikesh\Documents\VS_CODE\LLM_RAG_practices\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sample Documents

In [3]:
documents = [
    "Transformers use attention mechanism to process sequences.",
    "RAG stands for Retrieval Augmented Generation.",
    "Embeddings convert text into numerical vectors.",
    "FAISS is used for similarity search.",
    "Chunking splits large documents into smaller pieces."
]

# Chunking

In [4]:
#### simple Chunking
# def chunk_text(text, chunk_size=50):
#     words = text.split()
#     chunks = []

#     for i in range(0, len(words), chunk_size):
#         chunk = " ".join(words[i:i+chunk_size])
#         chunks.append(chunk)

#     return chunks





# Chunk Overlap
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

print(chunks)

['Transformers use attention mechanism to process sequences.', 'RAG stands for Retrieval Augmented Generation.', 'Embeddings convert text into numerical vectors.', 'FAISS is used for similarity search.', 'Chunking splits large documents into smaller pieces.']


# Create Embeddings

In [5]:
emd = SentenceTransformer('all-MiniLM-L6-v2')
chunk_emd = emd.encode(chunks)

c:\Users\Pratikesh\Documents\VS_CODE\LLM_RAG_practices\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pratikesh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3881.63it/

# Store in FAISS

In [6]:
dim = chunk_emd.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(np.array(chunk_emd))

# MY own pdf

In [8]:
import PyPDF2

def load_pdf(file_path):
    text = ""
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text




pdf_text = load_pdf("C:/Users/Pratikesh/Downloads/Pratikesh_Howale_DS.pdf")





chunks = chunk_text(pdf_text, chunk_size=100)
chunk_embeddings = emd.encode(chunks)





index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings))

# Retriever Function

In [9]:
def ret(query,top_k=3):
    q_emd = emd.encode([query])
    distances, indices = index.search(np.array(q_emd),top_k)
    results = [chunks[i] for i in indices[0]]
    return results

# generator(LLM)

In [10]:
gen = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length = 100
)

c:\Users\Pratikesh\Documents\VS_CODE\LLM_RAG_practices\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pratikesh\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4040.23it/s]
The tied weights

# RAG Pipeline

In [12]:
def rag_f(query):
    ret_doc = ret(query)
    context = " ".join(ret_doc)

    prompt = f"""
Use ONLY the context below. If answer not found, say "I don't know".

Context: {context}

Question: {query}
"""

    results = gen(prompt)[0]['generated_text']

    return results


# Testing

In [14]:
query = "whats name?"
answer = rag_f(query)

print(f"ans:{answer}")

ans:
Use ONLY the context below. If answer not found, say "I don't know".

Context: Internship (30 Days) (Certificate )  Deep Learning A -Z 2024: Neural Networks, AI ( Certificate )  2024 Tableau Certified Data Analyst Training ( Certificate )  Data Analysis Essentials Using Excel ( Certificate )  The Ultimate Flask Course ( Certificate )  The Data Analyst Course: Complete Data Analyst Bootcamp (Certificate ) PERSONAL D ETAILS  Date of Birth : 19th February 1999  Languages Known : English, Hindi and Marathi  Address : Narhe, Pune, Maharashtra Pratikesh Howale Data Scientist Pratikeshhowale@gmail.com 8788961243 LinkedIn PROFILE SUMMARY  I have completed extensive training in Machine Learning and Data Analysis, with 6 months of practical experience .  Experienced as Data Scientist showcasing proficiency in Machine Learning and a Data Scientist fundamentals, leading to successful project outcomes  Strong programming skills and experience in data visualization make me an asset t